In [9]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
from scipy.spatial import cKDTree
import seaborn as sns
from utils import construct_path, read_feather_gcs
from matplotlib.colors import TwoSlopeNorm,  SymLogNorm
pd.set_option('display.max_rows', 200)

In [10]:
### LOAD DATA
DATASET     = "malecns_09"
SUBSET_NAME = "optic"
DATA_PATH   = "connectome_data"
pin_data    = 'pin_data'
out_dir     = Path("results/connectome")
dataset_base = DATASET.split("_")[0]  # "malecns"

if SUBSET_NAME is not None:
    edgelist_path = (
        f"{DATA_PATH}/{dataset_base}/"
        f"{dataset_base}_{SUBSET_NAME}_{DATASET}_{SUBSET_NAME}_simple_edgelist.feather"
    )
else:
    edgelist_path = construct_path(DATA_PATH, DATASET, "edgelist_simple")
    print("Using full dataset")

meta_path = construct_path(DATA_PATH, DATASET, "meta")
# print(f"Meta path: {meta_path}")
print(f"Edgelist path: {edgelist_path}")

#Column pins from Nern et al 2025. Nature
#NOTE: !!!!!!
#Coordinate systems
# X Y Z
# Data from neuprint uses x y z coordinates. Here is a place to describe that coordinate system.

# Hex1 Hex2
# Omatidia in the Drosophila compound eye are arranged in what resembles a hexagonal grid. 
# Each individual omatidia can be uniqueliy identified along the 3 hexagonal axis hex1, hex2, and hex3. 
# In fact, any two of these coordinates is enough to identify the location. 
# The hex1 hex2 coordinates used in the neuprint database and in our analysis are positive numbers. 
# The origin of the coordinate system is arbitrarily located outside of the eyemap.

# P Q V
# A biologically more relevant coordinate system uses the center of the eye as the origin 
# of the P and Q coordinate system. We define the hex1 hex2 coordintes [18 19] as our origin for P Q.

# The H and V-axis use the same origin with V representing the virtual hex3. 
# The H axis is somewhat parallel to the perceived horizon.

meta = read_feather_gcs(meta_path, gcs_fs=None)
meta["malecns_09_id"] = meta["malecns_09_id"].astype("int64")
ct1_node = meta.query("cell_type == 'CT1' and side =='left'").copy().reset_index(drop=True)
ct1_node.rename(columns={"optic_lobe_hex_1": "hex_1",
                               "optic_lobe_hex_2": "hex_2"
                               }, inplace=True)
edgelist = read_feather_gcs(edgelist_path, gcs_fs=None)
edgelist = edgelist[edgelist["count"] > 3].copy().reset_index(drop=True)


Edgelist path: connectome_data/malecns/malecns_optic_malecns_09_optic_simple_edgelist.feather
✓ Loaded 165,114 rows
✓ Loaded 6,629,951 rows


In [11]:
cols_to_keep= ['malecns_09_id', 'side', 'cell_type','neurotransmitter_predicted', 'neurotransmitter_score',
                'hex_1', 'hex_2',
                ]
ct1_node = ct1_node[cols_to_keep].copy()


In [12]:
data_dir = Path("connectome_data/processed")
col_nodes = pd.read_feather(data_dir / "columnar_nodes.feather")
T4_T5_nodes = pd.read_feather(data_dir / "T4_T5_home_columns.feather")
# nodes_df = pd.concat([col_nodes, T4_T5_nodes, ct1_node], ignore_index=True).drop_duplicates("malecns_09_id", keep="last").copy()
nodes_df = pd.concat([col_nodes, T4_T5_nodes], 
                     ignore_index=True).drop_duplicates(
                         "malecns_09_id", keep="last").copy() #lets skip ct1 to not double count when creating edgelist


ct1_edgelist = pd.read_feather(data_dir / "ct1_columnar_edges.feather").drop(columns=['ct1_direction', 'neuropil'])
ct1_edgelist['pre_side'] = 'right'
ct1_edgelist['post_side'] = 'right'

## Annotated edgelist

In [13]:
# ids = set(nodes_df["malecns_09_id"])
ids = nodes_df["malecns_09_id"].unique()
edges_df = edgelist.query("pre in @ids and post in @ids").copy().reset_index(drop=True)
edges_df["pre"] = edges_df["pre"].astype("int64")
edges_df["post"] = edges_df["post"].astype("int64")

post_info = nodes_df[
    ["malecns_09_id", "cell_type", "hex_1", "hex_2", "p", "q", "side"]
].rename(
    columns={
        "malecns_09_id": "post",
        "cell_type": "post_cell_type",
        "hex_1": "target_hex_1",
        "hex_2": "target_hex_2",
            "p": "target_p",
            "q": "target_q",
        "side": "post_side",
    }
    )

pre_info = nodes_df[
    ["malecns_09_id", "cell_type", "hex_1", "hex_2", "p", "q", "side", "neurotransmitter_predicted", "neurotransmitter_score"]
].rename(
    columns={
        "malecns_09_id": "pre",
        "cell_type": "pre_cell_type",
        "hex_1": "source_hex_1",
        "hex_2": "source_hex_2",
        "p": "source_p",
        "q": "source_q",
        "side": "pre_side",
        "neurotransmitter_predicted": "edge_nt",
        "neurotransmitter_score": "edge_nt_score",
    }
)
edges_annot = edges_df.merge(
    post_info,
    on="post",
    how="left"
).merge(
    pre_info,
    on="pre",
    how="left"
)
edges_annot["dp"] =  edges_annot['target_p'] - edges_annot['source_p']
edges_annot["dq"] =  edges_annot['target_q'] - edges_annot['source_q']

front_cols = ['pre', 'post', 'pre_cell_type', 'post_cell_type', 'count', "dp", "dq"] #nt_predicted', 'nt_score']
# count	neuropil	source_hex_1	source_hex_2	source_p	source_q	target_hex_1	target_hex_2	target_p	target_q	dp	dq	edge_nt	edge_nt_score

# 2. Reorder by adding the rest of the columns automatically
edges_annot = edges_annot[front_cols + [c for c in edges_annot.columns if c not in front_cols]]

In [14]:
edges = pd.concat([edges_annot, ct1_edgelist], ignore_index=True).copy() #lets skip ct1 to not double count when creating edgelist
edges["sign"] = np.nan #Note some T4/5 and known nt synapses maybe incorrectly classified

edges.loc[edges["edge_nt"].eq("acetylcholine"), "sign"] = 1.0
edges.loc[edges["edge_nt"].isin(["gaba", "glutamate"]), "sign"] = -1.0
edges = edges[edges['count'] > 4.0]
edges

,pre,post,pre_cell_type,post_cell_type,count,dp,dq,norm,total_input,target_hex_1,target_hex_2,target_p,target_q,post_side,source_hex_1,source_hex_2,source_p,source_q,pre_side,edge_nt,edge_nt_score,sign
0,21598,31004,L2,L1,5,0.0,0.0,0.017065,293.0,32.0,21.0,14.0,2.0,right,32.0,21.0,14.0,2.0,right,acetylcholine,0.935050,1.0
1,76831,26444,L5,L2,10,0.0,0.0,0.034483,290.0,32.0,20.0,14.0,1.0,right,32.0,20.0,14.0,1.0,right,acetylcholine,0.957103,1.0
2,76831,23877,L5,L1,23,0.0,0.0,0.130682,176.0,32.0,20.0,14.0,1.0,right,32.0,20.0,14.0,1.0,right,acetylcholine,0.957103,1.0
3,545608,26444,C3,L2,69,0.0,0.0,0.237931,290.0,32.0,20.0,14.0,1.0,right,32.0,20.0,14.0,1.0,right,gaba,0.813528,-1.0
4,49458,21598,C3,L2,65,0.0,0.0,0.166240,391.0,32.0,21.0,14.0,2.0,right,32.0,21.0,14.0,2.0,right,gaba,0.818378,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178000,546619,10009,Tm1,CT1,7,0.0,0.0,NaN,NaN,27.0,17.0,9.0,-2.0,right,27.0,17.0,9.0,-2.0,right,acetylcholine,0.999533,1.0
178001,547644,10009,LC14b,CT1,6,NaN,NaN,NaN,NaN,32.0,18.0,14.0,-1.0,right,NaN,NaN,NaN,NaN,right,acetylcholine,0.990604,1.0
178005,554447,10009,Tm9,CT1,25,0.0,0.0,NaN,NaN,3.0,7.0,-15.0,-12.0,right,3.0,7.0,-15.0,-12.0,right,acetylcholine,0.996686,1.0
178007,579914,10009,Tm9,CT1,6,0.0,-1.0,NaN,NaN,7.0,5.0,-11.0,-14.0,right,7.0,6.0,-11.0,-13.0,right,acetylcholine,0.998023,1.0


In [15]:
nodes_final =  pd.concat([col_nodes, T4_T5_nodes, ct1_node], ignore_index=True).drop_duplicates("malecns_09_id", keep="last").copy()
nodes_final

,malecns_09_id,side,cell_type,neurotransmitter_predicted,neurotransmitter_score,hex_1,hex_2,p,q
0,100043,right,C2,gaba,0.528316,24.0,17.0,6.0,-2.0
1,100062,right,C2,unclear,0.398571,24.0,35.0,6.0,16.0
2,100134,right,C2,unclear,0.405942,22.0,31.0,4.0,12.0
3,100156,right,C2,unclear,0.467389,28.0,38.0,10.0,19.0
4,100229,right,C2,unclear,0.581083,27.0,13.0,9.0,-6.0
...,...,...,...,...,...,...,...,...,...
30299,163871,right,T5d,unclear,0.974427,31.0,39.0,13.0,20.0
30300,142354,right,T5d,unclear,0.974427,34.0,21.0,16.0,2.0
30301,160570,right,T5d,unclear,0.974427,27.0,39.0,9.0,20.0
30302,133675,right,T5d,unclear,0.974427,35.0,24.0,17.0,5.0


In [16]:
###Save: 
print('#'*100)
print('Saving data to: ', out_dir)
print('#'*100)

nodes_final.to_feather(
    out_dir/"nodes_OL_columnar.feather"
            )
edges.to_feather(
    out_dir/"edgelist_OL_columnar.feather"
)

####################################################################################################
Saving data to:  results/connectome
####################################################################################################
